----------------------------------------------------------------------------------------------
--------------------------------- ROW REMOVAL (NEIGHBORING DATASETS) -------------------------
----------------------------------------------------------------------------------------------


In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.compose import ColumnTransformer
import copy
import os

import import_ipynb
import importlib
import functions as fc
importlib.reload(fc)

print(os.getcwd())

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


/Users/elif/Desktop/Projects/privacy_in_ml/jupyter_notebooks


In [2]:
cc_train = pd.read_csv("../datasets/creditcard_train.csv")
cc_test = pd.read_csv("../datasets/creditcard_test.csv")

cc_train = cc_train.drop("ID", axis=1)
cc_test = cc_test.drop("ID", axis=1)

def get_dummies_all(X_train, X_test):
    """
    Converts all categorical variables in a DataFrame into dummy (one-hot encoded) variables.
    """
    categorical_cols = ["MARRIAGE", "SEX", "EDUCATION"]
    X_train_encoded = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
    X_test_encoded = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)
    return X_train_encoded, X_test_encoded

X_train = cc_train.drop('default.payment.next.month', axis=1)
X_test = cc_test.drop('default.payment.next.month', axis=1)
y_train = cc_train['default.payment.next.month']
y_test = cc_test['default.payment.next.month']

X_train, X_test = get_dummies_all(X_train, X_test)


In [3]:
numeric_features = [
    "LIMIT_BAL",
    "AGE",
    "BILL_AMT1",
    "BILL_AMT2",
    "BILL_AMT3",
    "BILL_AMT4",
    "BILL_AMT5",
    "BILL_AMT6",
    "PAY_AMT1",
    "PAY_AMT2",
    "PAY_AMT3",
    "PAY_AMT4",
    "PAY_AMT5",
    "PAY_AMT6",
]

ordinal_features = [
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6"
]

def preprocess(X_tr, X_te):
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_features),
            ("ord", StandardScaler(), ordinal_features)
        ],
        remainder="passthrough"
    )
    X_tr_sc = pd.DataFrame(
        preprocessor.fit_transform(X_tr),
        columns=preprocessor.get_feature_names_out(),
        index=X_tr.index
    )
    X_te_sc = pd.DataFrame(
        preprocessor.transform(X_te),
        columns=preprocessor.get_feature_names_out(),
        index=X_te.index
    )
    return X_tr_sc, X_te_sc

output_file = "../results/row_removal_cc.xlsx"
n_iter = 10
epsilon_values = [0.1, 1, 5, 10, 30, 50, 100]


### Baseline (full data, no removal, no perturbation)

In [4]:
X_train_sc, X_test_sc = preprocess(X_train, X_test)

baseline_model = LogisticRegression(
    penalty="l2",
    C=1,
    max_iter=1000,
    random_state=42
)
baseline_model.fit(X_train_sc, y_train)

print("Baseline (full data):")
fc.predict_binary(baseline_model, X_train_sc, y_train, conf_matrix=False)

fc.predict_binary_save_results(
    baseline_model,
    X_test_sc,
    y_test,
    conf_matrix=False,
    perturbation_type="Original",
    epsilon=0,
    row_id=None,
    output_file=output_file
)


Baseline (full data):
---------------------------------------
Accuracy: 0.8112
Precision: 0.7174
Recall: 0.2452
F1 Score: 0.3654
---------------------------------------
---------------------------------------
Accuracy: 0.8108
Precision: 0.6978
Recall: 0.2391
F1 Score: 0.3562
---------------------------------------
Saved results to ../results/row_removal_cc.xlsx


----------------------------------------------------------------------------------------------
-------------------------------- INPUT PERTURBATION (row removal) ----------------------------
----------------------------------------------------------------------------------------------

In [5]:
ranges = [[1000,1000000], [18,100], [-2,8], [-2,8], [-2,8], [-2,8], [-2,8],
       [-2,8], [-10000,500000], [-10000,500000], [-10000,500000], [-10000,500000],
       [-10000,500000],[-10000,500000], [0,250000], [0,250000], [0,250000],
       [0,250000], [0,250000], [0,250000], [0,1], [0,1], [0,1],
       [0,1], [0,1], [0,1], [0,1], [0,1]]
input_sensitivity = []
for r in ranges:
    input_sensitivity.append(r[1] - r[0])


In [6]:
row_rng = np.random.default_rng(42)  
np.random.seed(42)                    
for iteration in range(1, n_iter + 1):
    idx = row_rng.integers(0, X_train.shape[0])
    removed_label = X_train.index[idx]

    X_train_reduced = X_train.drop(index=removed_label)
    y_train_reduced = y_train.drop(index=removed_label)

    X_train_reduced_clipped = X_train_reduced.copy()
    for i, col in enumerate(X_train_reduced.columns):
        lower, upper = ranges[i]
        X_train_reduced_clipped[col] = X_train_reduced_clipped[col].clip(lower=lower, upper=upper)

    for e in epsilon_values:
        X_train_perturbed = X_train_reduced_clipped.copy()

        for i, var in enumerate(X_train_reduced_clipped.columns):
            eps_j = e  # The paper uses eps_j = e/d, but this leads to huge errors
            scale = input_sensitivity[i] / eps_j
            noise = np.random.laplace(loc=0, scale=scale, size=len(X_train_reduced_clipped))

            lower, upper = ranges[i]
            X_train_perturbed[var] = (
                X_train_reduced[var] + noise
            ).clip(lower=lower, upper=upper)

        X_train_perturbed_sc, X_test_perturbed_sc = preprocess(X_train_perturbed, X_test)

        input_model = copy.deepcopy(baseline_model)
        input_model.fit(X_train_perturbed_sc, y_train_reduced)

        print(f"Input - iteration {iteration}/{n_iter}, epsilon={e} - removed row {removed_label}")
        fc.predict_binary(input_model, X_train_perturbed_sc, y_train_reduced, conf_matrix=False)

        fc.predict_binary_save_results(
            input_model,
            X_test_perturbed_sc,
            y_test,
            conf_matrix=False,
            perturbation_type="Input",
            epsilon=e,
            row_id=removed_label,
            output_file=output_file
        )


Input - iteration 1/10, epsilon=0.1 - removed row 2142
---------------------------------------
Accuracy: 0.7782
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
---------------------------------------
---------------------------------------
Accuracy: 0.7812
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
---------------------------------------
Saved results to ../results/row_removal_cc.xlsx
Input - iteration 1/10, epsilon=1 - removed row 2142
---------------------------------------
Accuracy: 0.7782
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
---------------------------------------
---------------------------------------
Accuracy: 0.7812
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
---------------------------------------
Saved results to ../results/row_removal_cc.xlsx
Input - iteration 1/10, epsilon=5 - removed row 2142
---------------------------------------
Accuracy: 0.7819
Precision: 0.6261
Recall: 0.0415
F1 Score: 0.0779
---------------------------------------
--------

/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

Saved results to ../results/row_removal_cc.xlsx
Input - iteration 1/10, epsilon=10 - removed row 2142
---------------------------------------
Accuracy: 0.7910
Precision: 0.6549
Recall: 0.1223
F1 Score: 0.2061
---------------------------------------
---------------------------------------
Accuracy: 0.7952
Precision: 0.6858
Recall: 0.1181
F1 Score: 0.2014
---------------------------------------
Saved results to ../results/row_removal_cc.xlsx
Input - iteration 1/10, epsilon=30 - removed row 2142
---------------------------------------
Accuracy: 0.8044
Precision: 0.6955
Recall: 0.2098
F1 Score: 0.3224
---------------------------------------
---------------------------------------
Accuracy: 0.8080
Precision: 0.6785
Recall: 0.2331
F1 Score: 0.3469
---------------------------------------
Saved results to ../results/row_removal_cc.xlsx
Input - iteration 1/10, epsilon=50 - removed row 2142
---------------------------------------
Accuracy: 0.8074
Precision: 0.7002
Recall: 0.2303
F1 Score: 0.3466

/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

Input - iteration 2/10, epsilon=5 - removed row 18574
---------------------------------------
Accuracy: 0.7814
Precision: 0.6056
Recall: 0.0404
F1 Score: 0.0757
---------------------------------------
---------------------------------------
Accuracy: 0.7828
Precision: 0.6190
Recall: 0.0198
F1 Score: 0.0384
---------------------------------------
Saved results to ../results/row_removal_cc.xlsx
Input - iteration 2/10, epsilon=10 - removed row 18574
---------------------------------------
Accuracy: 0.7937
Precision: 0.6759
Recall: 0.1336
F1 Score: 0.2231
---------------------------------------
---------------------------------------
Accuracy: 0.7970
Precision: 0.6779
Recall: 0.1379
F1 Score: 0.2291
---------------------------------------
Saved results to ../results/row_removal_cc.xlsx
Input - iteration 2/10, epsilon=30 - removed row 18574
---------------------------------------
Accuracy: 0.8055
Precision: 0.7019
Recall: 0.2133
F1 Score: 0.3271
---------------------------------------
-----

...

Input - iteration 8/10, epsilon=1 - removed row 16736
---------------------------------------
Accuracy: 0.7782
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
---------------------------------------
---------------------------------------
Accuracy: 0.7812
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
---------------------------------------
Saved results to ../results/row_removal_cc.xlsx
Input - iteration 8/10, epsilon=5 - removed row 16736
---------------------------------------
Accuracy: 0.7826
Precision: 0.6431
Recall: 0.0443
F1 Score: 0.0830
---------------------------------------
---------------------------------------
Accuracy: 0.7830
Precision: 0.6279
Recall: 0.0206
F1 Score: 0.0398
---------------------------------------
Saved results to ../results/row_removal_cc.xlsx
Input - iteration 8/10, epsilon=10 - removed row 16736
---------------------------------------
Accuracy: 0.7909
Precision: 0.6529
Recall: 0.1219
F1 Score: 0.2055
---------------------------------------
------

/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

Input - iteration 9/10, epsilon=5 - removed row 4835
---------------------------------------
Accuracy: 0.7824
Precision: 0.6138
Recall: 0.0517
F1 Score: 0.0953
---------------------------------------
---------------------------------------
Accuracy: 0.7832
Precision: 0.6429
Recall: 0.0206
F1 Score: 0.0399
---------------------------------------
Saved results to ../results/row_removal_cc.xlsx
Input - iteration 9/10, epsilon=10 - removed row 4835
---------------------------------------
Accuracy: 0.7935
Precision: 0.6746
Recall: 0.1332
F1 Score: 0.2225
---------------------------------------
---------------------------------------
Accuracy: 0.7985
Precision: 0.6857
Recall: 0.1462
F1 Score: 0.2411
---------------------------------------
Saved results to ../results/row_removal_cc.xlsx
Input - iteration 9/10, epsilon=30 - removed row 4835
---------------------------------------
Accuracy: 0.8057
Precision: 0.7048
Recall: 0.2130
F1 Score: 0.3272
---------------------------------------
--------

/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

Saved results to ../results/row_removal_cc.xlsx
Input - iteration 10/10, epsilon=10 - removed row 2260
---------------------------------------
Accuracy: 0.7925
Precision: 0.6660
Recall: 0.1293
F1 Score: 0.2165
---------------------------------------
---------------------------------------
Accuracy: 0.7977
Precision: 0.6787
Recall: 0.1432
F1 Score: 0.2365
---------------------------------------
Saved results to ../results/row_removal_cc.xlsx
Input - iteration 10/10, epsilon=30 - removed row 2260
---------------------------------------
Accuracy: 0.8046
Precision: 0.6926
Recall: 0.2142
F1 Score: 0.3272
---------------------------------------
---------------------------------------
Accuracy: 0.8077
Precision: 0.6779
Recall: 0.2308
F1 Score: 0.3443
---------------------------------------
Saved results to ../results/row_removal_cc.xlsx
Input - iteration 10/10, epsilon=50 - removed row 2260
---------------------------------------
Accuracy: 0.8072
Precision: 0.6977
Recall: 0.2311
F1 Score: 0.3

----------------------------------------------------------------------------------------------
-------------------------------- OUTPUT PERTURBATION (row removal) ---------------------------
----------------------------------------------------------------------------------------------

In [7]:
row_rng = np.random.default_rng(42)   # reset: same 10 rows as the Input section above
np.random.seed(42)

for iteration in range(1, n_iter + 1):
    idx = row_rng.integers(0, X_train.shape[0])
    removed_label = X_train.index[idx]

    X_train_reduced = X_train.drop(index=removed_label)
    y_train_reduced = y_train.drop(index=removed_label)

    X_train_reduced_sc, X_test_reduced_sc = preprocess(X_train_reduced, X_test)

    reduced_model = copy.deepcopy(baseline_model)
    reduced_model.fit(X_train_reduced_sc, y_train_reduced)

    n = X_train_reduced.shape[0]
    lambda_reg = 1 / reduced_model.C
    output_sensitivity = 2 / (lambda_reg * n)   # matches creditcard_dataset.ipynb's Output section

    for e in epsilon_values:
        coef_noise = np.random.laplace(loc=0, scale=output_sensitivity / e, size=reduced_model.coef_.shape)
        intercept_noise = np.random.laplace(loc=0, scale=output_sensitivity / 3, size=reduced_model.intercept_.shape)

        output_model = copy.deepcopy(reduced_model)
        output_model.coef_ = reduced_model.coef_ + coef_noise
        output_model.intercept_ = reduced_model.intercept_ + intercept_noise

        print(f"Output - iteration {iteration}/{n_iter}, epsilon={e} - removed row {removed_label}")
        fc.predict_binary(output_model, X_train_reduced_sc, y_train_reduced, conf_matrix=False)

        fc.predict_binary_save_results(
            output_model,
            X_test_reduced_sc,
            y_test,
            conf_matrix=False,
            perturbation_type="Output",
            epsilon=e,
            row_id=removed_label,
            output_file=output_file
        )


Output - iteration 1/10, epsilon=0.1 - removed row 2142
---------------------------------------
Accuracy: 0.8110
Precision: 0.7171
Recall: 0.2438
F1 Score: 0.3639
---------------------------------------
---------------------------------------
Accuracy: 0.8102
Precision: 0.6951
Recall: 0.2361
F1 Score: 0.3525
---------------------------------------
Saved results to ../results/row_removal_cc.xlsx
Output - iteration 1/10, epsilon=1 - removed row 2142
---------------------------------------
Accuracy: 0.8111
Precision: 0.7166
Recall: 0.2452
F1 Score: 0.3653
---------------------------------------
---------------------------------------
Accuracy: 0.8108
Precision: 0.6978
Recall: 0.2391
F1 Score: 0.3562
---------------------------------------
Saved results to ../results/row_removal_cc.xlsx
Output - iteration 1/10, epsilon=5 - removed row 2142
---------------------------------------
Accuracy: 0.8110
Precision: 0.7165
Recall: 0.2450
F1 Score: 0.3651
---------------------------------------
-----

----------------------------------------------------------------------------------------------
-------------------------------- INTERNAL PERTURBATION (row removal) -------------------------
----------------------------------------------------------------------------------------------

In [8]:
row_rng = np.random.default_rng(42)   # reset: same 10 rows as Input/Output above

for iteration in range(1, n_iter + 1):
    idx = row_rng.integers(0, X_train.shape[0])
    removed_label = X_train.index[idx]

    X_train_reduced = X_train.drop(index=removed_label)
    y_train_reduced = y_train.drop(index=removed_label)

    X_train_reduced_sc, X_test_reduced_sc = preprocess(X_train_reduced, X_test)

    data_norm = np.max(np.linalg.norm(np.asarray(X_train_reduced_sc), axis=1))

    print(f"Internal - iteration {iteration}/{n_iter} - removed row {removed_label}, data_norm={data_norm:.4f}")

    fc.internal_perturbation_save_results(
        "cc_row_removal",
        X_train_reduced_sc,
        y_train_reduced,
        X_test_reduced_sc,
        y_test,
        epsilon_values=epsilon_values,
        data_norm=data_norm,
        C=baseline_model.C,
        bivariate=True,
        row_id=removed_label,
        output_file=output_file,
        perturbation_type="Internal"
    )


Internal - iteration 1/10 - removed row 2142, data_norm=94.5970
---------------------------------------
Epsilon: 0.1
Accuracy: 0.6265
Precision: 0.2434
Recall: 0.3351
F1 Score: 0.2820
Time: 0.01 s
---------------------------------------
---------------------------------------
Epsilon: 1
Accuracy: 0.7000
Precision: 0.3087
Recall: 0.2993
F1 Score: 0.3039
Time: 0.01 s
---------------------------------------
---------------------------------------
Epsilon: 5
Accuracy: 0.7497
Precision: 0.3967
Recall: 0.2765
F1 Score: 0.3259
Time: 0.03 s
---------------------------------------
---------------------------------------
Epsilon: 10
Accuracy: 0.7443
Precision: 0.3883
Recall: 0.2925
F1 Score: 0.3336
Time: 0.07 s
---------------------------------------


/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


---------------------------------------
Epsilon: 30
Accuracy: 0.8038
Precision: 0.6435
Recall: 0.2323
F1 Score: 0.3414
Time: 0.09 s
---------------------------------------


/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-pac

---------------------------------------
Epsilon: 50
Accuracy: 0.8078
Precision: 0.6786
Recall: 0.2315
F1 Score: 0.3453
Time: 0.04 s
---------------------------------------
---------------------------------------
Epsilon: 100
Accuracy: 0.8117
Precision: 0.7084
Recall: 0.2369
F1 Score: 0.3550
Time: 0.02 s
---------------------------------------
Saved results to ../results/row_removal_cc.xlsx
Internal - iteration 2/10 - removed row 18574, data_norm=94.5970
---------------------------------------
Epsilon: 0.1
Accuracy: 0.6265
Precision: 0.2434
Recall: 0.3351
F1 Score: 0.2820
Time: 0.01 s
---------------------------------------
---------------------------------------
Epsilon: 1
Accuracy: 0.7000
Precision: 0.3087
Recall: 0.2993
F1 Score: 0.3039
Time: 0.01 s
---------------------------------------
---------------------------------------
Epsilon: 5
Accuracy: 0.7495
Precision: 0.3963
Recall: 0.2765
F1 Score: 0.3257
Time: 0.03 s
---------------------------------------


...

/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


---------------------------------------
Epsilon: 1
Accuracy: 0.7000
Precision: 0.3087
Recall: 0.2993
F1 Score: 0.3039
Time: 0.01 s
---------------------------------------
---------------------------------------
Epsilon: 5
Accuracy: 0.7497
Precision: 0.3967
Recall: 0.2765
F1 Score: 0.3259
Time: 0.03 s
---------------------------------------
---------------------------------------
Epsilon: 10
Accuracy: 0.7443
Precision: 0.3878
Recall: 0.2909
F1 Score: 0.3325
Time: 0.08 s
---------------------------------------
---------------------------------------
Epsilon: 30
Accuracy: 0.8038
Precision: 0.6429
Recall: 0.2331
F1 Score: 0.3421
Time: 0.05 s
---------------------------------------


/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-pac

---------------------------------------
Epsilon: 50
Accuracy: 0.8073
Precision: 0.6748
Recall: 0.2308
F1 Score: 0.3439
Time: 0.03 s
---------------------------------------
---------------------------------------
Epsilon: 100
Accuracy: 0.8113
Precision: 0.7071
Recall: 0.2353
F1 Score: 0.3531
Time: 0.02 s
---------------------------------------
Saved results to ../results/row_removal_cc.xlsx
Internal - iteration 10/10 - removed row 2260, data_norm=94.5970
---------------------------------------
Epsilon: 0.1
Accuracy: 0.6265
Precision: 0.2434
Recall: 0.3351
F1 Score: 0.2820
Time: 0.01 s
---------------------------------------
---------------------------------------
Epsilon: 1
Accuracy: 0.7000
Precision: 0.3087
Recall: 0.2993
F1 Score: 0.3039
Time: 0.01 s
---------------------------------------
---------------------------------------
Epsilon: 5
Accuracy: 0.7495
Precision: 0.3963
Recall: 0.2765
F1 Score: 0.3257
Time: 0.03 s
---------------------------------------
---------------------------

/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
